# All the questions are generating from gemini Api


In [ ]:
import google.generativeai as genai
import json
import random
import sys

# Configure Gemini API (replace with your actual API key)
genai.configure(api_key="api beka technical coordinator contact madu")
model = genai.GenerativeModel("gemini-1.5-flash")

# Updated aptitude categories
aptitude_categories = [
    "Numerical Ability",
    "Logical Reasoning",
    "Quantitative Aptitude",
    "Verbal Reasoning"
]

# Minimal fallback questions for aptitude categories (5 per category)
fallback_aptitude_questions = {
    "Numerical Ability": [
        {"question": "What is 20% of 150?", "options": ["30", "25", "35", "40"], "answer": "30"},
        {"question": "If a car travels 60 km in 1.5 hours, what is its speed in km/h?", "options": ["30", "40", "50", "60"], "answer": "40"},
        {"question": "What is the value of 5! (5 factorial)?", "options": ["120", "100", "110", "130"], "answer": "120"},
        {"question": "If x = 5 and y = 3, what is x^y?", "options": ["125", "15", "8", "25"], "answer": "125"},
        {"question": "What is the square root of 144?", "options": ["10", "12", "14", "16"], "answer": "12"}
    ],
    "Logical Reasoning": [
        {"question": "If all cats are animals, and some animals are dogs, can we conclude that some cats are dogs?", "options": ["Yes", "No", "Maybe", "Cannot be determined"], "answer": "No"},
        {"question": "Complete the series: 2, 4, 6, 8, ?", "options": ["10", "12", "14", "16"], "answer": "10"},
        {"question": "If A is taller than B, and B is taller than C, who is the tallest?", "options": ["A", "B", "C", "Cannot be determined"], "answer": "A"},
        {"question": "If today is Monday, what day is it after 10 days?", "options": ["Wednesday", "Thursday", "Friday", "Saturday"], "answer": "Thursday"},
        {"question": "If all roses are flowers, and some flowers are red, can we conclude that some roses are red?", "options": ["Yes", "No", "Maybe", "Cannot be determined"], "answer": "Cannot be determined"}
    ],
    "Quantitative Aptitude": [
        {"question": "A train travels 120 km in 2 hours. What is its speed?", "options": ["50 km/h", "60 km/h", "70 km/h", "80 km/h"], "answer": "60 km/h"},
        {"question": "If 5 workers can build a wall in 10 days, how many days will 10 workers take?", "options": ["5 days", "10 days", "15 days", "20 days"], "answer": "5 days"},
        {"question": "What is the simple interest on $1000 at 5% per annum for 2 years?", "options": ["$100", "$200", "$300", "$400"], "answer": "$100"},
        {"question": "If a shirt costs $20 after a 20% discount, what was the original price?", "options": ["$24", "$25", "$26", "$27"], "answer": "$25"},
        {"question": "What is the area of a rectangle with length 5 cm and width 3 cm?", "options": ["15 cm²", "16 cm²", "17 cm²", "18 cm²"], "answer": "15 cm²"}
    ],
    "Verbal Reasoning": [
        {"question": "Choose the word that is most similar to 'happy':", "options": ["Sad", "Joyful", "Angry", "Tired"], "answer": "Joyful"},
        {"question": "Complete the analogy: Cat is to kitten as dog is to ?", "options": ["Puppy", "Cub", "Foal", "Calf"], "answer": "Puppy"},
        {"question": "Identify the misspelled word:", "options": ["Recieve", "Believe", "Achieve", "Deceive"], "answer": "Recieve"},
        {"question": "What is the antonym of 'expand'?", "options": ["Contract", "Extend", "Increase", "Grow"], "answer": "Contract"},
        {"question": "Choose the correct sentence: ", "options": ["He go to school.", "He goes to school.", "He going to school.", "He gone to school."], "answer": "He goes to school."}
    ]
}

# Minimal fallback domain questions
fallback_domain_questions = [
    {"question": "What is the primary role of an engineer?", "options": ["Design systems", "Manage finances", "Conduct surveys", "Write reports"], "answer": "Design systems"},
    {"question": "What does CAD stand for in engineering?", "options": ["Computer-Aided Design", "Computer-Algorithm Design", "Circuit Analysis Design", "Control Automation Design"], "answer": "Computer-Aided Design"},
    {"question": "Which material is commonly used in structural engineering?", "options": ["Steel", "Wood", "Plastic", "Glass"], "answer": "Steel"},
    {"question": "What is the purpose of a prototype in engineering?", "options": ["Test concepts", "Final product", "Marketing tool", "Cost estimation"], "answer": "Test concepts"},
    {"question": "What is the unit of power in engineering?", "options": ["Watt", "Joule", "Newton", "Volt"], "answer": "Watt"}
]

# Generate questions using Gemini API
def generate_questions(prompt, fallback, num_questions=5, retries=2):
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            text = response.text.strip().replace("```json", "").replace("```", "")
            questions = json.loads(text) if text else []
            # Standardize options to be a list
            for q in questions:
                if 'options' in q and isinstance(q['options'], dict):
                    q['options'] = [q['options'][str(i)] for i in range(1, 5)]
            if isinstance(questions, list) and len(questions) >= num_questions:
                return questions[:num_questions]
            print(f"Invalid Gemini response (attempt {attempt+1}), retrying...")
        except Exception as e:
            print(f"Error generating questions (attempt {attempt+1}): {e}")
    print("Failed to generate questions, using fallback")
    return random.sample(fallback, min(num_questions, len(fallback)))

# Generate aptitude questions for a category
def generate_aptitude_questions(category):
    prompt = f"""
    Generate 5 unique multiple-choice questions with 4 options each on the aptitude category: {category}.
    Each question should include:
    - A question
    - Four answer options
    - The correct answer
    Format each question as a dictionary with keys: 'question', 'options', 'answer'.
    Return the result as a JSON string containing a list of 5 dictionaries.
    """
    fallback = fallback_aptitude_questions.get(category, fallback_domain_questions)
    return generate_questions(prompt, fallback)

# Generate domain-specific questions
def generate_domain_questions(domain):
    prompt = f"""
    Generate 5 unique multiple-choice questions with 4 options each for a {domain} technical aptitude test.
    Each question must be relevant to {domain}.
    Each question should include:
    - A question
    - Four answer options
    - The correct answer
    Format each question as a dictionary with keys: 'question', 'options', 'answer'.
    Return the result as a JSON string containing a list of 5 dictionaries.
    """
    return generate_questions(prompt, fallback_domain_questions)

# Main aptitude test function
def run_aptitude_test():
    # Get domain from user
    print("Enter your engineering domain (e.g., CSE, Mechanical, Civil, ECE):")
    selected_domain = input().strip().upper()
    if not selected_domain:
        print("Error: Domain cannot be empty!")
        sys.exit(1)

    # Use the updated aptitude categories
    selected_categories = aptitude_categories
    print(f"Aptitude categories: {selected_categories}")

    # Generate aptitude questions (20 total: 5 from each of the 4 categories)
    questions = []
    score = {category: 0 for category in selected_categories}
    score["domain"] = 0
    for category in selected_categories:
        category_questions = generate_aptitude_questions(category)
        questions.extend([{**q, "category": category} for q in category_questions])

    # Generate domain questions (5 total)
    domain_questions = generate_domain_questions(selected_domain)
    questions.extend([{**q, "category": "domain"} for q in domain_questions])

    # Shuffle questions
    random.shuffle(questions)

    # Run quiz with improved input handling
    user_answers = []
    for i, q in enumerate(questions, 1):
        print(f"\nQuestion {i}: {q['question']}")
        for j, opt in enumerate(q['options'], 1):
            print(f"{j}. {opt}")
        while True:
            try:
                user_input = input("Enter option number (1-4, or 'q' to quit): ").strip()
                if user_input.lower() == 'q':
                    print("Quitting test.")
                    sys.exit(0)
                ans = int(user_input)
                if 1 <= ans <= 4:
                    user_answers.append(q["options"][ans-1])
                    break
                else:
                    print("Invalid input! Enter a number between 1 and 4.")
            except ValueError:
                print("Invalid input! Enter a number between 1 and 4.")

    # Evaluate results
    for q, ans in zip(questions, user_answers):
        if ans and q["answer"] == ans:
            score[q["category"]] += 1

    # Display results
    total = sum(score.values())
    print("\n=== Results ===")
    print(f"Total Score: {total}/25")
    for category in selected_categories:
        print(f"{category}: {score[category]}/5")
    print(f"{selected_domain} Domain: {score['domain']}/5")

if __name__ == "__main__":
    run_aptitude_test()

# Only domain based question are generating from gemini api


In [1]:
import csv
import random
import sys
import json
import google.generativeai as genai
import uuid

In [ ]:
# Configure the Gemini API 
try:
    genai.configure(api_key="Nin api key aki run madu")
except Exception as e:
    print(f"Failed to configure Gemini API: {e}")
    sys.exit(1)

In [3]:
# List available models to debug model error
try:
    models = genai.list_models()
    available_models = [model.name for model in models]
    print("Available models:", available_models)
except Exception as e:
    print(f"Failed to list models: {e}. Using fallback questions.")
    available_models = []

# Select a suitable model, accounting for 'models/' prefix
preferred_models = ['models/gemini-1.5-pro', 'models/gemini-1.5-flash', 'models/gemini-pro']
model_name = None
for model in preferred_models:
    if model in available_models:
        model_name = model
        break
if model_name:
    try:
        model = genai.GenerativeModel(model_name.split('/')[-1])  # Remove 'models/' prefix
        print(f"Using model: {model_name}")
    except Exception as e:
        print(f"Failed to initialize model {model_name}: {e}. Using fallback questions.")
        model_name = None
else:
    print("No suitable model found. Using fallback questions only.")

Available models: ['models/chat-bison-001', 'models/text-bison-001', 'models/embedding-gecko-001', 'models/gemini-1.0-pro-vision-latest', 'models/gemini-pro-vision', 'models/gemini-1.5-pro-latest', 'models/gemini-1.5-pro-001', 'models/gemini-1.5-pro-002', 'models/gemini-1.5-pro', 'models/gemini-1.5-flash-latest', 'models/gemini-1.5-flash-001', 'models/gemini-1.5-flash-001-tuning', 'models/gemini-1.5-flash', 'models/gemini-1.5-flash-002', 'models/gemini-1.5-flash-8b', 'models/gemini-1.5-flash-8b-001', 'models/gemini-1.5-flash-8b-latest', 'models/gemini-1.5-flash-8b-exp-0827', 'models/gemini-1.5-flash-8b-exp-0924', 'models/gemini-2.5-pro-exp-03-25', 'models/gemini-2.5-pro-preview-03-25', 'models/gemini-2.5-flash-preview-04-17', 'models/gemini-2.0-flash-exp', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-exp-image-generation', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-2.0-flash-lite-preview-02-05', 'models/gemin

In [4]:
# Define supported domains if needed extra hako

supported_domains = ['CSE', 'MECHANICAL', 'CIVIL', 'ECE', 'ELECTRICAL', 
                     'CHEMICAL', 'AEROSPACE', 'BIOTECHNOLOGY', 'INDUSTRIAL', 'ENVIRONMENTAL']

In [5]:
# esta edre extra questions, extra domain hako yar beda andru ninge...
fallback_domain_questions = {
    'CSE': [
        {"question": "What is the time complexity of binary search?", "options": ["O(n)", "O(log n)", "O(n log n)", "O(1)"], "answer": "B", "topic": "CSE"},
        {"question": "Which data structure uses LIFO?", "options": ["Queue", "Stack", "Array", "Linked List"], "answer": "B", "topic": "CSE"},
        {"question": "What does CPU stand for?", "options": ["Central Processing Unit", "Computer Power Unit", "Control Process Utility", "Central Power Utility"], "answer": "A", "topic": "CSE"},
        {"question": "What is the purpose of an operating system?", "options": ["Gaming", "Manage hardware", "Word processing", "Web browsing"], "answer": "B", "topic": "CSE"},
        {"question": "Which is a high-level programming language?", "options": ["Assembly", "Machine Code", "Python", "Binary"], "answer": "C", "topic": "CSE"}
    ],
    'MECHANICAL': [
        {"question": "What is the first law of thermodynamics?", "options": ["Energy conservation", "Entropy increase", "Heat transfer", "Work done"], "answer": "A", "topic": "MECHANICAL"},
        {"question": "What is the unit of force?", "options": ["Newton", "Joule", "Watt", "Pascal"], "answer": "A", "topic": "MECHANICAL"},
        {"question": "What is a lever?", "options": ["Simple machine", "Complex engine", "Electrical device", "Hydraulic pump"], "answer": "A", "topic": "MECHANICAL"},
        {"question": "What does RPM stand for?", "options": ["Rotations Per Minute", "Revolutions Per Meter", "Radians Per Minute", "Rate Per Motion"], "answer": "A", "topic": "MECHANICAL"},
        {"question": "What is shear stress?", "options": ["Force per unit area", "Thermal expansion", "Volume change", "Pressure drop"], "answer": "A", "topic": "MECHANICAL"}
    ],
    'CIVIL': [
        {"question": "What is the primary material in concrete?", "options": ["Cement", "Steel", "Wood", "Glass"], "answer": "A", "topic": "CIVIL"},
        {"question": "What is a beam?", "options": ["Structural element", "Decorative piece", "Electrical conduit", "Plumbing pipe"], "answer": "A", "topic": "CIVIL"},
        {"question": "What does RCC stand for?", "options": ["Reinforced Concrete Cement", "Rapid Construction Code", "Residential Civil Contract", "Road Construction Crew"], "answer": "A", "topic": "CIVIL"},
        {"question": "What is the purpose of a foundation?", "options": ["Support structure", "Aesthetic design", "Insulation", "Ventilation"], "answer": "A", "topic": "CIVIL"},
        {"question": "What is a retaining wall?", "options": ["Soil support", "Decorative barrier", "Roof structure", "Window frame"], "answer": "A", "topic": "CIVIL"}
    ],
    'ECE': [
        {"question": "What does LED stand for?", "options": ["Light Emitting Diode", "Low Energy Device", "Linear Electric Diode", "Light Energy Detector"], "answer": "A", "topic": "ECE"},
        {"question": "What is an amplifier used for?", "options": ["Increase signal strength", "Reduce noise", "Store energy", "Filter light"], "answer": "A", "topic": "ECE"},
        {"question": "What is the unit of resistance?", "options": ["Ohm", "Volt", "Ampere", "Watt"], "answer": "A", "topic": "ECE"},
        {"question": "What is a transistor?", "options": ["Switching device", "Power supply", "Cooling fan", "Display unit"], "answer": "A", "topic": "ECE"},
        {"question": "What does AC stand for?", "options": ["Alternating Current", "Active Circuit", "Analog Control", "Amplified Current"], "answer": "A", "topic": "ECE"}
    ],
    'ELECTRICAL': [
        {"question": "What is Ohm's Law?", "options": ["V=IR", "P=VI", "E=mc²", "F=ma"], "answer": "A", "topic": "ELECTRICAL"},
        {"question": "What is a conductor?", "options": ["Material that allows current", "Insulating material", "Magnetic field", "Thermal barrier"], "answer": "A", "topic": "ELECTRICAL"},
        {"question": "What is the unit of power?", "options": ["Watt", "Joule", "Volt", "Ampere"], "answer": "A", "topic": "ELECTRICAL"},
        {"question": "What is a fuse?", "options": ["Safety device", "Power generator", "Signal amplifier", "Light source"], "answer": "A", "topic": "ELECTRICAL"},
        {"question": "What does DC stand for?", "options": ["Direct Current", "Dynamic Circuit", "Digital Control", "Distributed Current"], "answer": "A", "topic": "ELECTRICAL"}
    ],
    'CHEMICAL': [
        {"question": "What is H2O?", "options": ["Water", "Hydrogen gas", "Oxygen gas", "Carbon dioxide"], "answer": "A", "topic": "CHEMICAL"},
        {"question": "What is a catalyst?", "options": ["Speeds up reaction", "Slows down reaction", "Stops reaction", "Heats solution"], "answer": "A", "topic": "CHEMICAL"},
        {"question": "What is pH a measure of?", "options": ["Acidity", "Temperature", "Pressure", "Density"], "answer": "A", "topic": "CHEMICAL"},
        {"question": "What is an element?", "options": ["Pure substance", "Mixture", "Compound", "Solution"], "answer": "A", "topic": "CHEMICAL"},
        {"question": "What gas do plants use?", "options": ["Carbon dioxide", "Oxygen", "Nitrogen", "Helium"], "answer": "A", "topic": "CHEMICAL"}
    ],
    'AEROSPACE': [
        {"question": "What is lift in aviation?", "options": ["Upward force", "Downward force", "Side thrust", "Drag"], "answer": "A", "topic": "AEROSPACE"},
        {"question": "What powers most aircraft?", "options": ["Jet engines", "Solar panels", "Wind turbines", "Electric motors"], "answer": "A", "topic": "AEROSPACE"},
        {"question": "What is the speed of sound called?", "options": ["Mach", "Warp", "Sonic", "Pulse"], "answer": "A", "topic": "AEROSPACE"},
        {"question": "What is an airfoil?", "options": ["Wing shape", "Engine type", "Fuel tank", "Landing gear"], "answer": "A", "topic": "AEROSPACE"},
        {"question": "What does NASA stand for?", "options": ["National Aeronautics and Space Administration", "North American Science Agency", "New Aerospace Standards Authority", "National Aviation Systems Alliance"], "answer": "A", "topic": "AEROSPACE"}
    ],
    'BIOTECHNOLOGY': [
        {"question": "What is DNA?", "options": ["Genetic material", "Protein", "Enzyme", "Hormone"], "answer": "A", "topic": "BIOTECHNOLOGY"},
        {"question": "What is cloning?", "options": ["Copying organisms", "Editing genes", "Synthesizing proteins", "Growing plants"], "answer": "A", "topic": "BIOTECHNOLOGY"},
        {"question": "What is a gene?", "options": ["DNA segment", "Cell organelle", "Blood type", "Virus"], "answer": "A", "topic": "BIOTECHNOLOGY"},
        {"question": "What is PCR used for?", "options": ["Amplify DNA", "Sequence proteins", "Grow cells", "Measure pH"], "answer": "A", "topic": "BIOTECHNOLOGY"},
        {"question": "What is an enzyme?", "options": ["Biological catalyst", "Structural protein", "Energy source", "Transport molecule"], "answer": "A", "topic": "BIOTECHNOLOGY"}
    ],
    'INDUSTRIAL': [
        {"question": "What is lean manufacturing?", "options": ["Waste reduction", "Mass production", "High inventory", "Slow processing"], "answer": "A", "topic": "INDUSTRIAL"},
        {"question": "What is a supply chain?", "options": ["Product flow", "Employee roster", "Machine maintenance", "Financial audit"], "answer": "A", "topic": "INDUSTRIAL"},
        {"question": "What is quality control?", "options": ["Product standards", "Marketing strategy", "Cost reduction", "Staff training"], "answer": "A", "topic": "INDUSTRIAL"},
        {"question": "What is automation?", "options": ["Machine operation", "Manual labor", "Design phase", "Sales pitch"], "answer": "A", "topic": "INDUSTRIAL"},
        {"question": "What is an assembly line?", "options": ["Production sequence", "Storage unit", "Shipping dock", "Office layout"], "answer": "A", "topic": "INDUSTRIAL"}
    ],
    'ENVIRONMENTAL': [
        {"question": "What is renewable energy?", "options": ["Sustainable source", "Fossil fuel", "Nuclear power", "Coal energy"], "answer": "A", "topic": "ENVIRONMENTAL"},
        {"question": "What is global warming?", "options": ["Temperature rise", "Ozone depletion", "Soil erosion", "Water scarcity"], "answer": "A", "topic": "ENVIRONMENTAL"},
        {"question": "What is recycling?", "options": ["Reusing materials", "Burning waste", "Landfilling", "Mining"], "answer": "A", "topic": "ENVIRONMENTAL"},
        {"question": "What gas causes the greenhouse effect?", "options": ["Carbon dioxide", "Oxygen", "Nitrogen", "Helium"], "answer": "A", "topic": "ENVIRONMENTAL"},
        {"question": "What is biodiversity?", "options": ["Species variety", "Air purity", "Water depth", "Soil type"], "answer": "A", "topic": "ENVIRONMENTAL"}
    ]
}

In [6]:
# extra beka contact technical coordinator
fallback_verbal_questions = [
    {"question": "What is the synonym of 'big'?", "options": ["Small", "Large", "Tiny", "Short"], "answer": "B", "topic": "Verbal Communication"},
    {"question": "Which word is misspelled?", "options": ["Receive", "Belive", "Achieve", "Deceive"], "answer": "B", "topic": "Verbal Communication"},
    {"question": "What is the antonym of 'happy'?", "options": ["Sad", "Joyful", "Excited", "Pleased"], "answer": "A", "topic": "Verbal Communication"},
    {"question": "Complete the analogy: Big is to small as tall is to ?", "options": ["High", "Short", "Long", "Wide"], "answer": "B", "topic": "Verbal Communication"},
    {"question": "Choose the correct sentence:", "options": ["He go to school.", "He goes to school.", "He going to school.", "He gone to school."], "answer": "B", "topic": "Verbal Communication"}
]

In [7]:
# Function to load questions from CSV
def load_questions_from_csv(filename, topic):
    questions = []
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            reader = csv.DictReader(file, delimiter=';')
            for row in reader:
                options = [row['Option A'], row['Option B'], row['Option C'], row['Option D']]
                question = {
                    'question': row['Question'],
                    'options': options,
                    'answer': row['Answer'],
                    'topic': topic
                }
                if is_valid_question(question):
                    questions.append(question)
                else:
                    print(f"Skipped invalid CSV question in {filename}: {question}")
    except FileNotFoundError:
        print(f"Error: File {filename} not found.")
        sys.exit(1)
    return questions

In [8]:
def is_valid_option(option):
    if not isinstance(option, str) or option.strip() == '':
        return False
    # Reject options that are just letters or placeholders
    invalid_options = ['A', 'B', 'C', 'D', 'a', 'b', 'c', 'd']
    return option.strip().upper() not in invalid_options

In [9]:
# Update the is_valid_question function to check option content
def is_valid_question(question):
    if not all(key in question for key in ['question', 'options', 'answer']):
        return False
    if not isinstance(question['options'], list) or len(question['options']) != 4:
        return False
    if question['answer'] not in ['A', 'B', 'C', 'D']:
        return False
    # Check if all options are meaningful
    for opt in question['options']:
        if not is_valid_option(opt):
            return False
    return True

In [10]:
# Modified generate_domain_questions function
def generate_domain_questions(domain, num_questions=5):
    if not model_name:
        print(f"No API model available. Using fallback questions for {domain}.")
        return random.sample(fallback_domain_questions[domain], num_questions)
    
    prompt = f"""
    Generate {num_questions} multiple-choice questions with 4 options each for {domain} engineering aptitude test.
    Each question must:
    - Be relevant to {domain} engineering.
    - Have a clear, concise question.
    - Provide four distinct, meaningful options (not placeholders like 'A', 'B', 'C', 'D').
    - Label options as A, B, C, D in the answer field.
    - Include the correct answer as 'A', 'B', 'C', or 'D'.
    Return the result as a JSON string containing a list of {num_questions} dictionaries with keys: 'question' (string), 'options' (list of 4 strings), 'answer' (string: 'A', 'B', 'C', or 'D').
    Example:
    [
        {{
            "question": "What is the time complexity of binary search?",
            "options": ["O(n)", "O(log n)", "O(n log n)", "O(1)"],
            "answer": "B"
        }}
    ]
    """
    try:
        response = model.generate_content(prompt)
        text = response.text.strip().replace("```json", "").replace("```", "")
        # Log the raw response for debugging
        # print(f"Raw API response for {domain} questions: {text}")
        questions = json.loads(text)
        parsed_questions = []
        for q in questions:
            if is_valid_question(q):
                parsed_questions.append({
                    'question': q['question'],
                    'options': q['options'],
                    'answer': q['answer'],
                    'topic': domain
                })
            else:
                print(f"Invalid question skipped: {q}")
        if len(parsed_questions) >= num_questions:
            return parsed_questions[:num_questions]
        else:
            print(f"Generated only {len(parsed_questions)} valid domain questions. Using fallback.")
            fallback = random.sample(fallback_domain_questions[domain], num_questions - len(parsed_questions))
            return parsed_questions + fallback
    except Exception as e:
        print(f"Failed to generate domain-specific questions: {e}. Using fallback questions.")
        return random.sample(fallback_domain_questions[domain], num_questions)

In [11]:
def generate_verbal_questions(num_questions=5):
    if not model_name:
        print("No API model available. Using fallback questions for Verbal Communication.")
        return random.sample(fallback_verbal_questions, num_questions)
    
    prompt = f"""
    Generate {num_questions} multiple-choice questions with 4 options each for Verbal Communication aptitude test.
    Questions should test vocabulary, grammar, analogies, sentence correction, or comprehension skills.
    Each question must:
    - Be clear and concise.
    - Provide four distinct, meaningful options (not placeholders like 'A', 'B', 'C', 'D').
    - Label options as A, B, C, D in the answer field.
    - Include the correct answer as 'A', 'B', 'C', or 'D'.
    Return the result as a JSON string containing a list of {num_questions} dictionaries with keys: 'question' (string), 'options' (list of 4 strings), 'answer' (string: 'A', 'B', 'C', or 'D').
    Example:
    [
        {{
            "question": "What is the synonym of 'big'?",
            "options": ["Small", "Large", "Tiny", "Short"],
            "answer": "B"
        }}
    ]
    """
    try:
        response = model.generate_content(prompt)
        text = response.text.strip().replace("```json", "").replace("```", "")
        # Log the raw response for debugging
        # print(f"Raw API response for Verbal Communication questions: {text}")
        questions = json.loads(text)
        parsed_questions = []
        for q in questions:
            if is_valid_question(q):
                parsed_questions.append({
                    'question': q['question'],
                    'options': q['options'],
                    'answer': q['answer'],
                    'topic': 'Verbal Communication'
                })
            else:
                print(f"Invalid question skipped: {q}")
        if len(parsed_questions) >= num_questions:
            return parsed_questions[:num_questions]
        else:
            print(f"Generated only {len(parsed_questions)} valid verbal questions. Using fallback.")
            fallback = random.sample(fallback_verbal_questions, num_questions - len(parsed_questions))
            return parsed_questions + fallback
    except Exception as e:
        print(f"Failed to generate Verbal Communication questions: {e}. Using fallback questions.")
        return random.sample(fallback_verbal_questions, num_questions)

In [12]:
# Load questions from datasets
try:
    general_aptitude_questions = load_questions_from_csv(r'I:\Hackathon_VVCE\App_for_aptitude\clean_general_aptitude_dataset.csv', 'General Aptitude')
    logical_reasoning_questions = load_questions_from_csv(r'I:\Hackathon_VVCE\App_for_aptitude\logical_reasoning_questions.csv', 'Logical Reasoning')
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure the CSV files are available.")
    sys.exit(1)


In [13]:
def main_matter():
    print("Welcome to the Aptitude Test.")
    print("You will answer 25 questions: 10 General Aptitude, 5 Logical Reasoning, 5 Verbal Communication, and 5 domain-specific.")
    print("For each question, enter the option number (1-4). Enter 'q' to quit.")
    while True:
        domain = input(f"Enter your engineering domain ({', '.join(supported_domains)}), or 'q' to quit: ").strip()
        
        if domain.lower() == 'q':
            sys.exit(1)  

        domain = domain.upper()
        print(f"Entered domain: '{domain}'")  # Debug output

        if domain in supported_domains:
            break
        else:
            print("Invalid domain. Try again or enter 'q' to quit.")

    # Select questions
    general_questions = random.sample(general_aptitude_questions, 10)
    logical_questions = random.sample(logical_reasoning_questions, 5)
    verbal_questions = generate_verbal_questions(5)
    domain_questions = generate_domain_questions(domain, 5)

    # Combine and shuffle all questions
    all_questions = general_questions + logical_questions + verbal_questions + domain_questions
    random.shuffle(all_questions)

    # Run the quiz
    user_answers = []
    incorrect_questions = []
    for i, q in enumerate(all_questions, 1):
        print(f"\nQuestion {i}: {q['question']}")
        for j, opt in enumerate(q['options'], 1):
            print(f"{j}. {opt}")
        while True:
            user_input = input("Enter option number (1-4, or 'q' to quit): ").strip().lower()
            if user_input == 'q':
                print("Quitting test.")
                sys.exit(0)
            try:
                choice = int(user_input)
                if 1 <= choice <= 4:
                    chosen_letter = chr(ord('A') + choice - 1)
                    user_answers.append(chosen_letter)
                    if chosen_letter != q['answer']:
                        incorrect_questions.append({
                            'question': q['question'],
                            'user_answer': q['options'][choice-1],
                            'correct_answer': q['options'][ord(q['answer'])-ord('A')],
                            'topic': q['topic']
                        })
                    break
                else:
                    print("Invalid input! Enter a number between 1 and 4.")
            except ValueError:
                print("Invalid input! Enter a number between 1 and 4.")

    # Evaluate scores
    scores = {'General Aptitude': 0, 'Logical Reasoning': 0, 'Verbal Communication': 0, domain: 0}
    for q, ans in zip(all_questions, user_answers):
        if ans == q['answer']:
            scores[q['topic']] += 1

    # Display results
    print("\n--- Test Results ---")
    print(f"General Aptitude: {scores['General Aptitude']}/10")
    print(f"Logical Reasoning: {scores['Logical Reasoning']}/5")
    print(f"Verbal Communication: {scores['Verbal Communication']}/5")
    print(f"{domain}: {scores[domain]}/5")
    total_score = sum(scores.values())
    print(f"Total: {total_score}/25")

    # Display incorrect answers
    if incorrect_questions:
        print("\n--- Incorrect Answers ---")
        for idx, item in enumerate(incorrect_questions, 1):
            print(f"{idx}. Topic: {item['topic']}")
            print(f"   Question: {item['question']}")
            print(f"   Your Answer: {item['user_answer']}")
            print(f"   Correct Answer: {item['correct_answer']}")
    print("Thank you for taking the test!")

In [14]:
main_matter()

Welcome to the Aptitude Test.
You will answer 25 questions: 10 General Aptitude, 5 Logical Reasoning, 5 Verbal Communication, and 5 domain-specific.
For each question, enter the option number (1-4). Enter 'q' to quit.
Entered domain: 'CSE'

Question 1: A, B, and C can complete a task in 24, 6, and 12 days respectively. How many days will they take to finish the task if working together?
1. 1 day
2. 7/24 day
3. 3 3/7 days
4. 4 days
Invalid input! Enter a number between 1 and 4.

Question 2: A man standing at a point P is watching the top of a tower, which makes an angle of elevation of 30º with the man's eye. The man walks some distance towards the tower to watch its top and the angle of the elevation becomes 60º. What is the distance between the base of the tower and the point P?
1. 43 units
2. 8 units
3. 12 units
4. Data inadequate

Question 3: 1, 2, 6, 15, 31, 56, 91 – find the wrong number in the sequence.
1. 31
2. 91
3. 56
4. 15

Question 4: Which of the following is NOT a core com

In [1]:
import random
